In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Transpilar con gestores de pases

<details>
<summary><b>Package versions</b></summary>

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
La forma recomendada de transpilar un circuito es crear un gestor de pases por etapas y luego ejecutar su método `run` con el circuito como entrada. Esta página explica cómo transpilar circuitos cuánticos de esta manera.
## ¿Qué es un gestor de pases (por etapas)?
En el contexto del SDK de Qiskit, la transpilación se refiere al proceso de transformar un circuito de entrada a una forma adecuada para su ejecución en un dispositivo cuántico. La transpilación ocurre típicamente en una secuencia de pasos llamados pases de transpilador. El circuito es procesado por cada pase del transpilador en secuencia, donde la salida de un pase se convierte en la entrada del siguiente. Por ejemplo, un pase podría recorrer el circuito y fusionar todas las secuencias consecutivas de puertas de un solo qubit, y luego el siguiente pase podría sintetizar estas puertas en el conjunto de bases del dispositivo objetivo. Los pases de transpilador incluidos en Qiskit se encuentran en el módulo [qiskit.transpiler.passes](https://docs.quantum.ibm.com/api/qiskit/transpiler_passes).

Un gestor de pases es un objeto que almacena una lista de pases de transpilador y puede ejecutarlos sobre un circuito. Para crear un gestor de pases, inicializa un [`PassManager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManager) con una lista de pases de transpilador. Para ejecutar la transpilación sobre un circuito, llama al método [`run`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManager#run) con un circuito como entrada.

Un gestor de pases por etapas es un tipo especial de gestor de pases que representa un nivel de abstracción superior al de un gestor de pases normal. Mientras que un gestor de pases normal está compuesto por varios pases de transpilador, un gestor de pases por etapas está compuesto por varios *gestores de pases*. Esta es una abstracción útil porque la transpilación ocurre típicamente en etapas discretas, como se describe en [Etapas del transpilador](transpiler-stages), donde cada etapa está representada por un gestor de pases. Los gestores de pases por etapas están representados por la clase [`StagedPassManager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.StagedPassManager). El resto de esta página describe cómo crear y personalizar gestores de pases (por etapas).
## Generar un gestor de pases por etapas preconfigurado
Para crear un gestor de pases por etapas preconfigurado con valores predeterminados razonables, utiliza la función [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager):

In [1]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

backend = FakeSherbrooke()
pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

Para transpilar un circuito o una lista de circuitos con un gestor de pases, pasa el circuito o la lista de circuitos al método `run`. Hagamos esto sobre un circuito de dos qubits que consiste en una Hadamard seguida de dos puertas CX adyacentes:

In [2]:
from qiskit import QuantumRegister, QuantumCircuit

# Create a circuit
qubits = QuantumRegister(2, name="q")
circuit = QuantumCircuit(qubits)
a, b = qubits
circuit.h(a)
circuit.cx(a, b)
circuit.cx(b, a)

# Transpile it by calling the run method of the pass manager
transpiled = pass_manager.run(circuit)

# Draw it, excluding idle qubits from the diagram
transpiled.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/transpile-with-pass-managers/extracted-outputs/c1426e6c-f506-4938-8c0a-05198bae9746-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/transpile-with-pass-managers/extracted-outputs/dcc69b72-e13b-4df6-a51f-a5ef2108bae7-0.svg)

Consulta [Valores predeterminados y opciones de configuración de la transpilación](defaults-and-configuration-options) para obtener una descripción de los posibles argumentos de la función `generate_preset_pass_manager`. Los argumentos de `generate_preset_pass_manager` coinciden con los argumentos de la función [`transpile`](https://docs.quantum.ibm.com/api/qiskit/compiler#qiskit.compiler.transpile).

<CodeAssistantAdmonition tagLine="¿Tienes problemas para recordar los detalles del gestor de pases? Prueba a preguntarle al Asistente de Código de Qiskit." />

Si los gestores de pases preconfigurados no satisfacen tus necesidades, personaliza la transpilación creando gestores de pases (por etapas) o incluso pases de transpilación propios. El resto de esta página describe cómo crear gestores de pases. Para obtener instrucciones sobre cómo crear pases de transpilación, consulta [Escribe tu propio pase de transpilador](custom-transpiler-pass).

## Crea tu propio gestor de pases

El módulo [qiskit.transpiler.passes](https://docs.quantum.ibm.com/api/qiskit/transpiler_passes) incluye muchos pases de transpilador que se pueden usar para crear gestores de pases. Para crear un gestor de pases, inicializa un `PassManager` con una lista de pases. Por ejemplo, el siguiente código crea un pase de transpilador que fusiona puertas adyacentes de dos qubits y luego las sintetiza en una base de puertas [$R_y$](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.RYGate), [$R_z$](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.RZGate) y [$R_{xx}$](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.RXXGate).

In [3]:
from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import (
    Collect2qBlocks,
    ConsolidateBlocks,
    UnitarySynthesis,
)

basis_gates = ["rx", "ry", "rxx"]
translate = PassManager(
    [
        Collect2qBlocks(),
        ConsolidateBlocks(basis_gates=basis_gates),
        UnitarySynthesis(basis_gates),
    ]
)

Para demostrar este gestor de pases en acción, pruébalo sobre un circuito de dos qubits que consiste en una Hadamard seguida de dos puertas CX adyacentes:

In [4]:
from qiskit import QuantumRegister, QuantumCircuit

qubits = QuantumRegister(2, name="q")
circuit = QuantumCircuit(qubits)

a, b = qubits
circuit.h(a)
circuit.cx(a, b)
circuit.cx(b, a)

circuit.draw("mpl")

<Image src="../docs/images/guides/transpile-with-pass-managers/extracted-outputs/01317c54-68b5-4e41-893f-82ee223e22f0-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/transpile-with-pass-managers/extracted-outputs/bc208935-e63c-461b-90d0-a6f4cabc16b6-0.svg)

Para ejecutar el gestor de pases sobre el circuito, llama al método `run`.

In [5]:
translated = translate.run(circuit)
translated.draw("mpl")

<Image src="../docs/images/guides/transpile-with-pass-managers/extracted-outputs/019ad99b-bd38-4217-90ee-da43959dc8ad-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/transpile-with-pass-managers/extracted-outputs/adb5c242-5cba-4878-a00d-4ec47737d029-0.svg)

Para un ejemplo más avanzado que muestra cómo crear un gestor de pases para implementar la técnica de supresión de errores conocida como desacoplamiento dinámico, consulta [Crear un gestor de pases para desacoplamiento dinámico](dynamical-decoupling-pass-manager).

## Crea un gestor de pases por etapas

Un `StagedPassManager` es un gestor de pases compuesto por etapas individuales, donde cada etapa está definida por una instancia de `PassManager`. Puedes crear un `StagedPassManager` especificando las etapas deseadas. Por ejemplo, el siguiente código crea un gestor de pases por etapas con dos etapas, `init` y `translation`. La etapa `translation` está definida por el gestor de pases que se creó anteriormente.

In [6]:
from qiskit.transpiler import PassManager, StagedPassManager
from qiskit.transpiler.passes import UnitarySynthesis, Unroll3qOrMore

basis_gates = ["rx", "ry", "rxx"]
init = PassManager(
    [UnitarySynthesis(basis_gates, min_qubits=3), Unroll3qOrMore()]
)
staged_pm = StagedPassManager(
    stages=["init", "translation"], init=init, translation=translate
)

No hay límite en el número de etapas que puedes agregar a un gestor de pases por etapas.

Otra forma útil de crear un gestor de pases por etapas es comenzar con uno preconfigurado y luego reemplazar algunas de las etapas. Por ejemplo, el siguiente código genera un gestor de pases preconfigurado con nivel de optimización 3 y luego especifica una etapa `pre_layout` personalizada.

In [7]:
import numpy as np
from qiskit.circuit.library import HGate, PhaseGate, RXGate, TdgGate, TGate
from qiskit.transpiler.passes import InverseCancellation

pass_manager = generate_preset_pass_manager(3, backend)
inverse_gate_list = [
    HGate(),
    (RXGate(np.pi / 4), RXGate(-np.pi / 4)),
    (PhaseGate(np.pi / 4), PhaseGate(-np.pi / 4)),
    (TGate(), TdgGate()),
]
logical_opt = PassManager(
    [
        InverseCancellation(inverse_gate_list),
    ]
)

# Add pre-layout stage to run extra logical optimization
pass_manager.pre_layout = logical_opt

Las [funciones generadoras de etapas](https://docs.quantum.ibm.com/api/qiskit/transpiler_preset#stage-generator-functions) pueden ser útiles para construir gestores de pases personalizados.
Generan etapas que proporcionan funcionalidad común utilizada en muchos gestores de pases.
Por ejemplo, [`generate_embed_passmanager`](https://docs.quantum.ibm.com/api/qiskit/transpiler_preset#qiskit.transpiler.preset_passmanagers.generate_embed_passmanager) se puede usar para generar una etapa
que "incruste" un `Layout` inicial seleccionado desde un pase de diseño al dispositivo objetivo especificado.

## Próximos pasos
> **Tip:** - [Escribe un pase de transpilador personalizado](custom-transpiler-pass).
>     - [Crea un gestor de pases para desacoplamiento dinámico](dynamical-decoupling-pass-manager).
>     - Para aprender más sobre la función `generate_preset_passmanager`, consulta el tema [Configuración predeterminada y opciones de configuración de la transpilación](defaults-and-configuration-options).
>     - Prueba la guía [Comparar configuraciones del transpilador](/guides/circuit-transpilation-settings).
>     - Revisa la [documentación de la API del transpilador.](https://docs.quantum.ibm.com/api/qiskit/transpiler)